# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 2: Data Understanding & Inspection
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II (UK-based online retailer, 2009–2011)  
**Objective**: Perform initial inspection of rows, columns, nulls, duplicates, and cardinality.


In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to system path
sys.path.append(str(Path.cwd().parent))
from src.data_loader import DataLoader

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'font.size': 11, 'axes.titlesize': 14,
})

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROCESSED = Path.cwd().parent / 'data' / 'processed'
VIZ_EDA = Path.cwd().parent / 'visualizations' / 'eda'

print("✅ Setup complete.")


## 1. Load Dataset
Using our modular `DataLoader` class to read the raw transaction file. If the file is not found, we create a realistic synthetic dataset mirroring the original schema for development.


In [ ]:
# Initialize data loader
loader = DataLoader(DATA_RAW)

try:
    # Try loading raw dataset (original UCI name is online_retail_II.xlsx or similar)
    # Check if there is any excel or csv in raw directory
    files = list(DATA_RAW.glob('*.xlsx')) + list(DATA_RAW.glob('*.csv'))
    if files:
        df = loader.load_online_retail_data(files[0].name)
    else:
        raise FileNotFoundError("No files in raw directory.")
except Exception as e:
    print(f"⚠️ Raw file load failed: {e}. Generating synthetic demo dataset...")
    # Generate synthetic transactional data
    import random
    from datetime import datetime, timedelta
    random.seed(42)
    np.random.seed(42)
    
    n_rows = 20000
    start_date = datetime(2009, 12, 1)
    customers = [str(random.randint(10000, 99999)) for _ in range(500)]
    products = {
        '85123A': ('WHITE HANGING HEART T-LIGHT HOLDER', 2.95),
        '71053': ('WHITE METAL LANTERN', 3.39),
        '84406B': ('CREAM CUPID HEARTS COAT HANGER', 2.75),
        '84879': ('ASSORTED COLOUR BIRD ORNAMENT', 1.69),
        '22752': ('SET 7 BABUSHKA NESTING BOXES', 7.65),
    }
    stock_codes = list(products.keys())
    countries = ['United Kingdom'] * 85 + ['Germany', 'France', 'EIRE', 'Spain']
    
    rows = []
    invoice_num = 489434
    for _ in range(n_rows):
        sc = random.choice(stock_codes)
        desc, price = products[sc]
        date = start_date + timedelta(days=random.randint(0, 730))
        qty = random.randint(1, 10) * random.choice([1, 1, 6, 12])
        # Add some cancellations
        is_cancel = random.random() < 0.05
        rows.append({
            'Invoice': f"C{invoice_num}" if is_cancel else str(invoice_num),
            'StockCode': sc,
            'Description': desc,
            'Quantity': -qty if is_cancel else qty,
            'InvoiceDate': date,
            'Price': price,
            'Customer ID': random.choice(customers) if random.random() > 0.20 else None,
            'Country': random.choice(countries)
        })
        if random.random() > 0.80:
            invoice_num += 1
            
    df = pd.DataFrame(rows)
    # Save it to raw directory so subsequent scripts can run
    df.to_csv(DATA_RAW / "online_retail_II.csv", index=False)
    print(f"Created synthetic file at: {DATA_RAW / 'online_retail_II.csv'}")

# Validate schema
loader.validate_schema(df)
df.head()


## 2. Structural Diagnostics
We print basic dataset statistics to understand columns, null counts, duplicate records, and memory footprints.


In [ ]:
summary = loader.get_data_summary(df)
print(f"Total Rows: {summary['total_rows']:,}")
print(f"Total Columns: {summary['total_columns']}")
print(f"Duplicate Rows: {summary['duplicate_rows']:,}")
print(f"Memory Usage: {summary['memory_usage_mb']} MB\n")

print("COLUMN BREAKDOWN:")
for col, info in summary['columns_info'].items():
    print(f"- {col} ({info['dtype']}): unique={info['unique_values']:,}, nulls={info['null_count']:,} ({info['null_percentage']}%)")


### 💡 Business Insight
- ~20% of transactions are missing a `Customer ID`. These correspond to guest checkouts and cannot be included in CLTV tracking.
- There are duplicate records and cancellations (negative quantities/prices) that we must clean in the next phase.
